## Step1: Load the data

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv('hourly_weather_10_days.csv')
df.head()

,timestamp,temperature_C,humidity_%,wind_speed_kmph,pressure_hPa,visibility_km
0,2023-03-01 00:00:00,16.6,74.4,5.7,1012.5,9.5
1,2023-03-01 01:00:00,16.2,78.5,5.0,1012.1,10.3
2,2023-03-01 02:00:00,15.3,73.3,4.7,NaN,11.1
3,2023-03-01 03:00:00,15.8,72.4,1.3,1005.0,8.9
4,2023-03-01 04:00:00,20.9,70.6,6.8,1016.3,9.8


## Step 2: Basic Exploration

In [11]:
print(df.info())
print(df.describe())
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   timestamp        240 non-null    datetime64[ns]
 1   temperature_C    228 non-null    float64       
 2   humidity_%       224 non-null    float64       
 3   wind_speed_kmph  226 non-null    float64       
 4   pressure_hPa     223 non-null    float64       
 5   visibility_km    228 non-null    float64       
 6   date             240 non-null    object        
 7   hour             240 non-null    int32         
dtypes: datetime64[ns](1), float64(5), int32(1), object(1)
memory usage: 14.2+ KB
None
                 timestamp  temperature_C  humidity_%  wind_speed_kmph  \
count                  240     228.000000  224.000000       226.000000   
mean   2023-03-05 23:30:00      21.315789   66.795982        10.105310   
min    2023-03-01 00:00:00      11.500000   47.800000    

## Step 3: Handle Missing Values

In [12]:
col = ['timestamp','temperature_C','humidity_%','wind_speed_kmph','pressure_hPa','visibility_km']
df[col]=df[col].ffill()
print(df.isna().sum())

timestamp          0
temperature_C      0
humidity_%         0
wind_speed_kmph    0
pressure_hPa       0
visibility_km      0
date               0
hour               0
dtype: int64


## Step 4: Data Analysis

In [14]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
daily_avg_temp = df.groupby('date')['temperature_C'].mean()
print(daily_avg_temp)

summary = df[col].agg(['max', 'min', 'mean'])
print(summary)

avg_humidity_by_hour = df.groupby('hour')['humidity_%'].mean()
most_humid_hour = avg_humidity_by_hour.idxmax()
print(avg_humidity_by_hour)
print(f"Most humid hour on average: {most_humid_hour}:00")

date
2023-03-01    21.550000
2023-03-02    21.366667
2023-03-03    21.433333
2023-03-04    21.554167
2023-03-05    21.670833
2023-03-06    21.858333
2023-03-07    21.037500
2023-03-08    20.762500
2023-03-09    20.945833
2023-03-10    21.716667
Name: temperature_C, dtype: float64
               timestamp  temperature_C  humidity_%  wind_speed_kmph  \
max  2023-03-10 23:00:00      28.700000    88.10000        17.800000   
min  2023-03-01 00:00:00      11.500000    47.80000         1.300000   
mean 2023-03-05 23:30:00      21.389583    66.79875        10.107083   

      pressure_hPa  visibility_km  
max    1027.000000      12.600000  
min     998.100000       6.800000  
mean   1011.915833      10.002917  
hour
0     78.17
1     78.42
2     75.89
3     71.94
4     69.31
5     69.34
6     65.77
7     65.05
8     63.49
9     59.65
10    58.71
11    58.91
12    59.53
13    58.33
14    60.54
15    61.05
16    58.71
17    64.03
18    66.70
19    69.19
20    67.46
21    71.43
22    73.75
23   

## step 5:

In [16]:
temp = df['temperature_C'].to_numpy()
temp_matrix = temp.reshape((10, 24))

daily_min = temp_matrix.min(axis=1)
daily_max = temp_matrix.max(axis=1)
daily_mean = temp_matrix.mean(axis=1)

print("Daily Min Temps:\n", daily_min)
print("Daily Max Temps:\n", daily_max)
print("Daily Mean Temps:\n", daily_mean)

Daily Min Temps:
 [14.7 15.7 13.6 15.9 12.4 15.5 15.3 13.5 14.3 11.5]
Daily Max Temps:
 [28.2 28.7 25.7 27.1 24.9 26.2 25.9 26.  27.1 28.5]
Daily Mean Temps:
 [21.55       21.36666667 21.43333333 21.55416667 21.67083333 21.85833333
 21.0375     20.7625     20.94583333 21.71666667]


In [18]:
def normalize(matrix):
    mean = np.mean(matrix)
    std = np.std(matrix)
    return (matrix - mean) / std

normalized_temp_matrix = normalize(temp_matrix)
print(normalized_temp_matrix)

[[-1.38980772 -1.50587692 -1.76703261 -1.62194611 -0.14206386 -0.17108116
   0.40926482  0.32221292 -0.05501196  1.97619897  1.97619897  1.2217492
   1.1637146   0.58336862  1.80209518  0.69943781  0.17712643  0.67042051
   0.49631672  0.20614373 -0.92553093 -0.57732335 -0.49027145 -1.9411364 ]
 [-1.59292881 -1.65096341 -1.09963473 -0.40321955  0.17712643 -0.05501196
   0.38024752  0.03203993  0.64140321  2.12128547 -0.25813306  1.0766627
   0.20614373  0.90255891  0.35123022  0.14810913  0.72845511  0.72845511
   0.81550701  0.17712643 -0.43223685 -0.75142714 -1.41882502 -0.98356553]
 [-2.26032669 -1.65096341 -0.83847904 -1.59292881  1.25076649 -0.14206386
  -0.17108116 -0.20009846  0.26417833  1.25076649  0.9896108  -0.05501196
   1.1927319   0.14810913  0.29319563  0.9605935   0.87354161  0.87354161
   0.55435132  0.87354161  0.38024752 -0.14206386 -0.80946174 -1.73801531]
 [-1.36079042 -0.78044444 -1.01258283 -1.56391151  0.03203993  0.20614373
   1.65700868  0.67042051  0.14810913

In [20]:
wind = df['wind_speed_kmph'].to_numpy()
mask = wind > 15
high_wind = wind[mask]

#print("Mask:\n", mask)
print("High wind readings (>15 kmph):\n", high_wind)
print("Number of high-wind hours:", len(high_wind))

High wind readings (>15 kmph):
 [17.6 16.  16.5 16.3 16.7 15.8 17.8 15.1 16.3 15.2 17.  15.9 15.6 15.8
 15.4 15.6 16.3 15.3 16.2 16.9 15.3 15.2 15.5 17.4 17.4 15.4 15.4 16.5
 17.  15.7]
Number of high-wind hours: 30


## final step

In [21]:
def daily_summary(matrix):
    summaries = []
    for day_index in range(matrix.shape[0]):
        day_data = matrix[day_index]
        summary = {
            'day': day_index + 1,
            'min': day_data.min(),
            'max': day_data.max(),
            'mean': day_data.mean()
        }
        summaries.append(summary)
    return summaries

summaries = daily_summary(temp_matrix)

for s in summaries:
    print(s)

{'day': 1, 'min': np.float64(14.7), 'max': np.float64(28.2), 'mean': np.float64(21.55)}
{'day': 2, 'min': np.float64(15.7), 'max': np.float64(28.7), 'mean': np.float64(21.366666666666664)}
{'day': 3, 'min': np.float64(13.6), 'max': np.float64(25.7), 'mean': np.float64(21.433333333333334)}
{'day': 4, 'min': np.float64(15.9), 'max': np.float64(27.1), 'mean': np.float64(21.554166666666664)}
{'day': 5, 'min': np.float64(12.4), 'max': np.float64(24.9), 'mean': np.float64(21.670833333333334)}
{'day': 6, 'min': np.float64(15.5), 'max': np.float64(26.2), 'mean': np.float64(21.858333333333334)}
{'day': 7, 'min': np.float64(15.3), 'max': np.float64(25.9), 'mean': np.float64(21.037499999999998)}
{'day': 8, 'min': np.float64(13.5), 'max': np.float64(26.0), 'mean': np.float64(20.7625)}
{'day': 9, 'min': np.float64(14.3), 'max': np.float64(27.1), 'mean': np.float64(20.945833333333336)}
{'day': 10, 'min': np.float64(11.5), 'max': np.float64(28.5), 'mean': np.float64(21.71666666666667)}
